In [ ]:
import tifffile
import matplotlib.pyplot as plt
import numpy as np
import cv2
import einops
import pydicom
import torch
from glob import glob
import  pandas as pd
import json
from os.path import join as opj
import torchvision
from PIL import Image
from shapely.wkt import dumps as wktd
from functools import partial

import matplotlib.pyplot as plt
from matplotlib.collections import PatchCollection
from matplotlib.patches import Polygon as MplPolygon
from shapely.geometry import Polygon, MultiPolygon

import numpy as np
from skimage import measure
from shapely.geometry import Polygon, MultiPolygon
from shapely.affinity import translate
from shapely.strtree import STRtree
from torchvision.transforms.functional import adjust_contrast, adjust_brightness

import itertools

from ds.eval.inference_mrcnn import get_model as get_mrcnn_model, get_xform as get_ds_xform, get_stack_list, score_threshold_with_matrix_nms
from ds.eval.common import output_mask_to_images

from ts2.data.transforms import SRHBaseTransform
from torchsrh.datasets.db_improc import (get_srh_base_aug, process_read_srh)

from ts2.playgrounds.scbench.registration import register_mosaic

In [ ]:
def field_flatten(array: np.ndarray,
                  blur_type: str = 'gaussian',
                  kernel_size: int = 301) -> np.ndarray:
    """Function to correct for changes in average pixel intensity in the image.

    Peforms a gaussian or average blur of kernel_size on array, then divides
    array by blurred image and rescales by 10000.
    """

    # guassian blur
    if blur_type.lower() == 'gaussian':
        filter_img = cv2.GaussianBlur(array,
                                      ksize=(kernel_size, kernel_size),
                                      sigmaX=kernel_size)
    if blur_type.lower() == 'average':
        filter_img = cv2.blur(array, (kernel_size, kernel_size))

    # flatten image and rescale
    flat_image = array / filter_img
    flat_image *= 10000  # NOTE: this value should be changed with caution. Can affect AI performance.
    return flat_image

In [ ]:
def get_strip_names(slide_root):
    return pd.DataFrame({
        "ch2": sorted(glob(f"{slide_root}/strips/CH2/*.dcm")),
        "ch3": sorted(glob(f"{slide_root}/strips/CH3/*.dcm"))
    })

def get_strip_im(strip_names, strip_id):
    if (strip_id < 0) or (strip_id >= len(strip_names)):
        return torch.zeros(2, 6000, 1000)
    ch2 = pydicom.dcmread(strip_names["ch2"].iloc[strip_id]).pixel_array[:, :]
    ch3 = pydicom.dcmread(strip_names["ch3"].iloc[strip_id]).pixel_array[:, :]
    return torch.tensor(np.stack((ch2, ch3), axis=0))

def get_slide_strip_im(strip_names):
    ims = [get_strip_im(strip_names, i) for i in range(len(strip_names))]
    return torch.cat(ims, axis=2)

In [ ]:
def plot_polygons(image, polygons, ax, edgecolor='green', facecolor='none', linewidth=10):
    """
    Plots a list of Shapely Polygons and MultiPolygons on an image.

    Args:
        image (np.ndarray): The background image.
        polygons (list): List of Shapely Polygon or MultiPolygon objects.
        ax (matplotlib.axes.Axes): Matplotlib axis object to draw on.
        edgecolor (str): Color of the polygon edges.
        facecolor (str): Fill color for the polygons.
        linewidth (int): Width of the polygon edges.
    """
    patches = []
    
    for poly in polygons:
        if isinstance(poly, Polygon):
            # Single Polygon
            patches.append(MplPolygon(list(poly.exterior.coords), closed=True))
            for hole in poly.interiors:
                patches.append(MplPolygon(list(hole.coords), closed=True))
                
        elif isinstance(poly, MultiPolygon):
            # Multiple Polygons
            for p in poly.geoms:
                patches.append(MplPolygon(list(p.exterior.coords), closed=True))
                for hole in p.interiors:
                    patches.append(MplPolygon(list(hole.coords), closed=True))

    # Plot the image
    ax.imshow(image, origin='upper')
    
    # Create and add the PatchCollection
    coll = PatchCollection(
        patches,
        facecolor=facecolor,
        edgecolor=edgecolor,
        linewidths=linewidth
    )
    ax.add_collection(coll)
    ax.autoscale()        # Adjust view to data
    ax.set_aspect('equal', 'box')
    
    return ax

In [ ]:
def mask_to_polygon_bak(mask: np.ndarray, tolerance: float = 0.0) -> MultiPolygon:
    """
    Convert a binary mask → Shapely MultiPolygon.
    - mask: 2D bool/0–1 array
    - tolerance: simplify() tolerance in pixels (set 0 to skip)
    """
    # 1. extract all iso-contours at level=0.5 in one C call
    contours = measure.find_contours(mask.astype(np.uint8), level=0.5)

    # 2. build & optionally simplify polygons, drop degenerate ones
    polys = []
    for contour in contours:
        try:
            coords = contour[:, ::-1] # (row,col) -> (x,y)
            poly = Polygon(coords)
            if tolerance:
                poly = poly.simplify(tolerance, preserve_topology=True)

            if poly.is_valid and poly.area > 1e-6:
                polys.append(poly)
        except:
            print("WARNING - POLYGON NOT ADDED")

    return polys


def mask_to_polygon(mask: np.ndarray, tolerance: float = 0.0) -> MultiPolygon:
    """
    Convert a binary mask → Shapely MultiPolygon with support for multiple exteriors and multiple holes.
    - mask: 2D bool/0–1 array
    - tolerance: simplify() tolerance in pixels (set 0 to skip)
    """
    # Extract contours
    contours = measure.find_contours(mask.astype(np.uint8), level=0.5)
    
    # Convert to Shapely polygons
    polygons = []
    for contour in contours:
        coords = contour[:, ::-1]  # (row, col) -> (x, y)
        try:
            poly = Polygon(coords)
            if tolerance:
                poly = poly.simplify(tolerance, preserve_topology=True)
            if poly.is_valid and poly.area > 1e-6:
                polygons.append(poly)
        except:
            pass

    # Separate exteriors and holes
    exterior_polys = []
    hole_candidates = []

    # First pass: Identify polygons without any other polygons inside
    for poly in polygons:
        is_exterior = True
        for other_poly in polygons:
            if other_poly != poly and other_poly.contains(poly):
                is_exterior = False
                break
        
        if is_exterior:
            exterior_polys.append(poly)
        else:
            hole_candidates.append(poly)

    # Second pass: Assign holes to the appropriate exterior polygons
    final_polygons = []

    for ext_poly in exterior_polys:
        holes = [hole.exterior.coords for hole in hole_candidates if ext_poly.contains(hole)]
        final_polygons.append(Polygon(ext_poly.exterior, holes))

    return final_polygons


def mask_to_multipolygon(mask: np.ndarray, tolerance: float = 0.0) -> MultiPolygon:
    mp = MultiPolygon(mask_to_polygon(mask, tolerance))
    if mp.is_valid and mp.area > 1e-6:

        return mp
    else:
        return None

In [ ]:
def crop_with_zero_pad(img: torch.Tensor, box: torch.Tensor) -> torch.Tensor:
    """
    Safely crop a CHW image given a center‐based box, padding zeros where the box
    extends outside the image.

    Args:
        img   (torch.Tensor): shape (C, H, W)
        box   (torch.Tensor): shape (4,), (cx, cy, h, w), can be float or int

    Returns:
        torch.Tensor: cropped patch of shape (C, h, w)
    """
    # unpack and make ints
    cx, cy, h, w = box.tolist()
    cx, cy, h, w = int(cx), int(cy), int(h), int(w)

    C, H, W = img.shape

    # compute box corners in source coordinates
    x1 = cx - w // 2
    y1 = cy - h // 2
    x2 = x1 + w
    y2 = y1 + h

    # compute the intersection with the image
    x1_src = max(x1, 0)
    y1_src = max(y1, 0)
    x2_src = min(x2, W)
    y2_src = min(y2, H)

    # compute where that intersection maps into the output
    x1_dst = max(-x1, 0)
    y1_dst = max(-y1, 0)
    x2_dst = x1_dst + (x2_src - x1_src)
    y2_dst = y1_dst + (y2_src - y1_src)

    # create output and copy the valid region
    out = img.new_zeros((C, h, w))
    if (x2_src > x1_src) and (y2_src > y1_src):
        out[:, y1_dst:y2_dst, x1_dst:x2_dst] = img[:, y1_src:y2_src, x1_src:x2_src]

    return out

def crop_with_edge_pad(img: torch.Tensor, box: torch.Tensor) -> torch.Tensor:
    """
    Safely crop a CHW image given a center-based box, padding by
    replicating the border pixels where the box extends outside.

    Args:
        img   (torch.Tensor): shape (C, H, W)
        box   (torch.Tensor): shape (4,), (cx, cy, h, w)

    Returns:
        torch.Tensor: cropped patch of shape (C, h, w)
    """
    # unpack & to int
    cx, cy, h, w = map(int, box.tolist())
    C, H, W = img.shape

    # compute box corners
    x1 = cx - w // 2
    y1 = cy - h // 2
    x2 = x1 + w
    y2 = y1 + h

    # how much we need to pad on each side
    pad_left   = max(-x1, 0)
    pad_top    = max(-y1, 0)
    pad_right  = max(x2 - W, 0)
    pad_bottom = max(y2 - H, 0)

    # pad the image (batch dim added for F.pad)
    # pad format: (pad_left, pad_right, pad_top, pad_bottom)
    img_p = torch.nn.functional.pad(img.unsqueeze(0),
                  (pad_left, pad_right, pad_top, pad_bottom),
                  mode='replicate')[0]

    # adjust crop origin into padded image
    x1_p = x1 + pad_left
    y1_p = y1 + pad_top

    return img_p[:, y1_p : y1_p + h, x1_p : x1_p + w]

def crop_with_edge_pad_np(img: np.ndarray, box: np.ndarray) -> np.ndarray:
    """
    Safely crop an HWC image given a center-based box, padding by
    replicating the border pixels where the box extends outside.

    Args:
        img   (np.ndarray): shape (H, W, C)
        box   (np.ndarray): shape (4,), (cx, cy, h, w)

    Returns:
        np.ndarray: cropped patch of shape (h, w, C)
    """
    cx, cy, h, w = map(int, box.tolist())
    H, W, C = img.shape

    # Compute crop corners
    x1 = cx - w // 2
    y1 = cy - h // 2
    x2 = x1 + w
    y2 = y1 + h

    # Compute required padding
    pad_left   = max(-x1, 0)
    pad_top    = max(-y1, 0)
    pad_right  = max(x2 - W, 0)
    pad_bottom = max(y2 - H, 0)

    # Pad image on H and W only
    if any(p > 0 for p in [pad_left, pad_right, pad_top, pad_bottom]):
        pad_width = (
            (pad_top, pad_bottom),   # H
            (pad_left, pad_right),   # W
            (0, 0),                  # C
        )
        img = np.pad(img, pad_width=pad_width, mode='edge')

    # Adjust crop origin into padded image
    x1_p = x1 + pad_left
    y1_p = y1 + pad_top

    return img[y1_p : y1_p + h, x1_p : x1_p + w, :]

def crop_im_from_centroid(im_raw, query, crop_size=300, final_size=300):
    
    c = query.centroid
    im_raw_crop = crop_with_edge_pad(im_raw,
                                 torch.tensor([int(c.x), int(c.y), crop_size, crop_size])).numpy()
    im_processed = torch.tensor(np.stack((
                field_flatten(im_raw_crop[0, ...]),
                field_flatten(im_raw_crop[1, ...])
            )).clip(min=0, max=65535))
    return torchvision.transforms.functional.center_crop(im_processed, output_size=final_size), (int(c.x), int(c.y))

In [ ]:
def get_annotated_files(slide_id, ttype, strip_width=900, strip_padding=50):
    new = tifffile.imread(f"images/todd_annotations/{ttype}/{slide_id}.tif")
    orig = tifffile.imread(f"images/scbench_{ttype}/{slide_id}.tiff")

    annot_with_bars = np.pad(new, ((strip_padding, strip_padding), (strip_padding, 0),(0,0)))[:,:-50,...]
    annot_tiles = einops.rearrange(annot_with_bars,
                                   "hp (w wp) c -> w hp wp c",
                                   wp=strip_width)

    orig_with_bars = np.pad(orig, ((strip_padding, strip_padding), (strip_padding, 0),(0,0)))[:,:-50,...]
    orig_tiles = einops.rearrange(orig_with_bars,
                                   "hp (w wp) c -> w hp wp c",
                                   wp=strip_width)

    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        (1 - np.all(new == orig, axis=2)).astype(np.uint8), 
        8)

    annotations = pd.DataFrame(np.hstack((centroids[1:], stats[1:])),
                               columns=["x", "y", "left", "top", "width", "height", "area"])

    annotations["strip_id"] = annotations["x"].apply(lambda x: int(x//strip_width))
    annotations["in_strip_x"] = annotations["x"].apply(lambda x:x%strip_width)
    annotations["in_strip_y"] = annotations["y"].apply(lambda x:x+50)

    return annot_with_bars, orig_with_bars, annotations

In [ ]:
def get_process_raw_strips_data(slide_id, db_root ="/nfs/turbo/umms-tocho/root_srh_db/UM/"):
    strip_im = get_slide_strip_im(get_strip_names(f"{db_root}/{slide_id.replace('.', '/')}"))
    new_strip_im = einops.rearrange(torch.tensor(register_mosaic(strip_im[0], strip_im[1])), "h w c -> c h w")
    new_strip_im_raw = einops.rearrange(
        einops.rearrange(new_strip_im, "c hp (w wp) -> c hp w wp", wp=1000)[:,:,:,:-100],
    "c hp w wp -> c hp (w wp)").clip(min=0, max=65535)
    return new_strip_im_raw

In [ ]:
def get_proc_cell_det_data(slide_id, celldet_root="/nfs/turbo/umms-tocho-snr/exp/chengjia/scsrh_repl_root_gen2/"):
    with open(f"{celldet_root}/{slide_id.replace('.', '_')}_raw_patch_list.json") as fd:
        patch_list = json.load(fd)
        
    celldet = torch.load(f"{celldet_root}/{slide_id.replace('.', '_')}_raw_pred.pt", map_location="cpu")
    #boxes = [i["boxes"][i["labels"] == 1] for i in celldet]
    contours = [[mask_to_multipolygon(i.numpy()) for i in j["masks"][j["labels"] == 1]] for j in celldet]
    contours = [[j  for j in i if j] for i in contours]
    patch_origin_coords = np.array([[int(j) for j in i.split("-")[-1].split("_")] for i in patch_list])
    patch_origin_coords_xy = np.vstack([patch_origin_coords[:,1]*0.9 + patch_origin_coords[:,3], patch_origin_coords[:,0] + patch_origin_coords[:,2]]).T
    patch_origin_coords_xyxy = np.hstack([patch_origin_coords_xy, patch_origin_coords_xy])
    
    #global_bbox = torch.cat([b+o for b, o in zip(boxes, patch_origin_coords_xyxy)])
    global_cells = list(itertools.chain(*[[translate(cc, xoff=i[0], yoff=i[1]) for cc in c]
                                             for c, i  in zip(contours, patch_origin_coords_xy)]))
    return global_cells



In [ ]:
def tie_break_matches(q, hits, global_cells):
    def dice(idx):
        poly = global_cells[idx]
        inter = poly.intersection(q).area
        return 2 * inter / (poly.area + q.area)
    
    top_hits = []
    for i in hits:
        if not any(
            i != j and global_cells[i].within(global_cells[j])
            for j in hits):
            top_hits.append(i)

    if not top_hits:
        top_hits = hits
        
    best = max(top_hits, key=dice)

    hits = [best]
    return hits

In [ ]:
def get_centroid_color_key(im, query):
    pt = query.representative_point()
    return tuple(im[int(pt.y), int(pt.x)].astype(int))

def is_on_edge(query, hcsz):
    c = query.centroid
    dx, dy = int(c.x)%300, int(c.y)%1000%300

    return ((np.abs(dx) < hcsz) | (np.abs(dx - 300) < hcsz) |
            (np.abs(dy) < hcsz) | (np.abs(dy - 300) < hcsz))

def match_queries(annot_with_bars, new_strip_im_raw, global_cells, queries, hcsz, final=False):
    polygons = []
    images = []
    annot_labels = []
    to_inf = []
    centroids = []
    tree = STRtree(global_cells)
    
    for qi, q in enumerate(queries):
        candidates = tree.query(q)
        hits = [poly
            for poly in candidates
            if global_cells[poly].intersects(q)
        ]

        celltype_q = cell_color_map.get(get_centroid_color_key(annot_with_bars, q), "ERROR")
        annot_labels.append(celltype_q)
        
        if len(hits) == 0:
            if final:
                imc_p = [crop_im_from_centroid(new_strip_im_raw, q, final_size=hcsz*2)]
                im_p = [i[0] for i in imc_p]
                c_p = [i[1] for i in imc_p]
                polygon_p = [None]
            else:
                to_inf.append((qi, *crop_im_from_centroid(new_strip_im_raw, q)))
                im_p, c_p, polygon_p = [], [], []

        elif celltype_q in {"glioma/tumor_cells", "metastatic/melanoma", "metastatic/adenocarcinoma", "metastatic/squamous_cell", "metastatic/sarcoma"} and q.area>1000:  # keep all valid

            if final:
                polygon_p = [global_cells[i] for i in hits]
                imc_p = [crop_im_from_centroid(new_strip_im_raw, q, final_size=hcsz*2) for q in polygon_p]
                im_p = [i[0] for i in imc_p]
                c_p = [i[1] for i in imc_p]
            else:
                hits = [h for h in hits if not is_on_edge(global_cells[h], hcsz)]    

                if len(hits) == 0:
                    to_inf.append((qi, *crop_im_from_centroid(new_strip_im_raw, q)))
                    im_p, c_p, polygon_p = [], [], []
                    print("ERROR - tumor_cell large region no match")

                else:
                    polygon_p = [global_cells[i] for i in hits]
                    imc_p = [crop_im_from_centroid(new_strip_im_raw, q, final_size=hcsz*2) for q in polygon_p]
                    im_p = [i[0] for i in imc_p]
                    c_p = [i[1] for i in imc_p]

        else: # keep unique
            if len(hits) > 1:
                hits = tie_break_matches(q, hits, global_cells)
                
            assert len(hits) == 1

            if not final and is_on_edge(q, hcsz):
                to_inf.append((qi, *crop_im_from_centroid(new_strip_im_raw, q)))
                im_p, c_p, polygon_p = [], [], []
            else:
                polygon_p = [global_cells[hits[0]]]
                imc_p = [crop_im_from_centroid(new_strip_im_raw, q, final_size=hcsz*2) for q in polygon_p]
                im_p = [i[0] for i in imc_p]
                c_p = [i[1] for i in imc_p]
    
        images.append(im_p)
        polygons.append(polygon_p)
        centroids.append(c_p)

    results = pd.DataFrame(
        {"polygons": polygons,
         "annot_labels": annot_labels,
         "images": images,
        "im_centroids": centroids})
    
    to_inf_images = torch.stack([i[1] for i in to_inf]).to(torch.float).contiguous() if to_inf else None
    to_inf_meta = pd.DataFrame(
        {"to_inf_target": [i[0] for i in to_inf],
        "crop_im_centroid": [i[2] for i in to_inf]})

    return results, to_inf_images, to_inf_meta

In [ ]:
def inference_on_images(images, model, aug, ds_cf):

    ims = [aug(i, {})[0] for i in images]
    # Inference on images
    with torch.inference_mode():
        results = []
        for im_b in torch.split(torch.stack(ims), 8):
            out = model(im_b.to("cuda"))
            for o in out:
                for k in o:
                    o[k] = o[k].detach().to("cpu")
            results.extend(out)
            torch.cuda.empty_cache()

    # Postprocessing")
    for i in range(len(results)):
        results[i] = score_threshold_with_matrix_nms(results[i], confidence_threshold=ds_cf["confidence_threshold"])
    
    return results

In [ ]:
def proc_one_slide(slide_id, ttype):
    print("  Loading annotation files")
    annot_with_bars, orig_with_bars, annotations = get_annotated_files(slide_id, ttype, strip_width=strip_width, strip_padding=strip_padding)
    queries = mask_to_polygon((1 - np.all(annot_with_bars == orig_with_bars, axis=2)))

    print("  Loading WSI strip files")
    new_strip_im_raw = get_process_raw_strips_data(slide_id)
    
    print("  Loading cell det results")
    global_cells = get_proc_cell_det_data(slide_id, celldet_root=celldet_root)
    
    print(f"  First stage matching -- {len(queries)} queries total")
    match, to_inf_images, to_inf_meta = match_queries(annot_with_bars, new_strip_im_raw, global_cells, queries, hcsz)
    
    print(f"  Running cell det inference for edge cases -- {len(to_inf_images)} cells total")
    reinf_results = inference_on_images(to_inf_images, model, aug, ds_cf)
    reinf_contours = [[mask_to_multipolygon(i.numpy()) for i in j["masks"][j["labels"] == 1]] for j in reinf_results]
    reinf_contours = [[j  for j in i if j] for i in reinf_contours]
    reinf_patch_origin_coords_xy = np.stack(to_inf_meta["crop_im_centroid"]) - hpsz
    reinf_global_cells = list(itertools.chain(*[[translate(cc, xoff=i[0], yoff=i[1]) for cc in c]
                                         for c, i  in zip(reinf_contours, reinf_patch_origin_coords_xy)]))

    print("  Rematching")
    missed_query_idx = set(to_inf_meta["to_inf_target"].tolist())
    missed_queries = [q for i,q in enumerate(queries) if i in missed_query_idx]
    reinf_match, _, reinf_meta = match_queries(annot_with_bars, new_strip_im_raw, reinf_global_cells, missed_queries, hcsz, final=True)

    print("  Processing results")
    for ((i, ti), (i, rm)) in zip(to_inf_meta.iterrows(), reinf_match.iterrows()):
        match.iloc[ti["to_inf_target"]] = rm  

    cols_to_explode = ['polygons', 'im_centroids', "images"]
    match_exploded = match.explode(cols_to_explode[0], ignore_index=True)
    for col in cols_to_explode[1:]:
        match_exploded[col] = match[col].explode(ignore_index=True)

    match_exploded["im_centroid_x"] = match_exploded["im_centroids"].apply(lambda x: x[0])
    match_exploded["im_centroid_y"] = match_exploded["im_centroids"].apply(lambda x: x[1])

    annot_ims = np.stack([
        crop_with_edge_pad_np(annot_with_bars, np.array([m["im_centroid_x"], m["im_centroid_y"], cell_sz, cell_sz]))
        for i, m in match_exploded.iterrows()])
    orig_ims = np.stack([
        crop_with_edge_pad_np(orig_with_bars, np.array([m["im_centroid_x"], m["im_centroid_y"], cell_sz, cell_sz]))
        for i, m in match_exploded.iterrows()])

    rgb_ims = (np.stack([
        einops.rearrange(adjust_brightness(adjust_contrast(transform(m["images"]), 2), 3), "c h w -> h w c")
        for i, m in match_exploded.iterrows()]) * 255).astype(np.uint8)

    all_viz = np.concatenate((annot_ims, orig_ims, rgb_ims), axis=2)
    
    match_exploded["polygons_wkt"] = match_exploded["polygons"].apply(partial(wktd, trim=True))
    match_exploded[["annot_labels", "im_centroid_x", "im_centroid_y", "polygons_wkt"]].to_csv(f"{slide_id}.csv", index=False)

    np.save(f"{slide_id}_viz.npy", all_viz)
    torch.save(torch.stack(match_exploded["images"].tolist()), f"{slide_id}.pt")

    return annot_with_bars, queries, match_exploded


In [ ]:
with open("cell_color_map.json") as fd:
    cell_color_pair = json.load(fd)
cell_color_map = {tuple(i[1]):i[0] for i in cell_color_pair}

In [ ]:
ds_cf = {
    "model_path": "/nfs/turbo/umms-tocho-snr/exp/chengjia/ds_reretrain/1e4/439971b1_Apr30-17-27-52_sd1000_ALLDATA_/models/ckpt-epoch19-step5500-loss0.00.ckpt",
    "classes": ["na", "nuclei","cyto", "rbc", "mp"],
    "subtracted_base": 5000,
    "confidence_threshold": 0.2
}

model = get_mrcnn_model(ds_cf["model_path"], num_classes=len(ds_cf["classes"]))
model = model.eval()
aug = get_ds_xform(subtracted_base=ds_cf["subtracted_base"])

In [ ]:
transform = SRHBaseTransform()
strip_width = 900
strip_padding=50
strip_height = "TODO"
cell_sz = 64
hcsz = cell_sz // 2
patch_sz=300
hpsz=patch_sz//2

In [ ]:
celldet_root="/nfs/mm-isilon/brainscans/dropbox/exp/chengjia/scsrh_repl_root_gen2/"

annoted_root = "/nfs/turbo/umms-tocho/code/chengjia/torchsrh2/ts2/playgrounds/scbench/images/todd_annotations/"
annoted_slides = glob(opj(annoted_root, "*/*.tif"))
def get_annoted_slide_info(x):
    cmp = x.removeprefix(annoted_root).removeprefix("/").split("/")
    return (cmp[1].removesuffix(".tif").removesuffix(".tiff"), cmp[0])

annoted_slides = [get_annoted_slide_info(i) for i in annoted_slides]


In [ ]:
processed_slides = [i.removeprefix("scbench_processed/").removesuffix(".pt")
                    for i in glob("scbench_processed/*.pt")]

In [ ]:
processed_slides = [i for i in annoted_slides if i[0] not in processed_slides]

In [ ]:
for ps in processed_slides:
    print(ps)
    
    try:
        proc_one_slide(*ps)
        print(f"OK -- {ps}")
    except:
        print(f"FAILED -- {ps}")
            


In [ ]:
pd.DataFrame(annoted_slides, columns=["mosaic", "ttype"]).to_csv("../../train/mosaic.csv", index=False)

In [ ]:
# debugging

In [ ]:
annot_with_bars, queries, match_exploded = proc_one_slide("NIO_UM_1808.3", "gbm")

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(20,20))
plot_polygons(annot_with_bars, queries + match_exploded["polygons"].tolist(), ax, edgecolor='green', facecolor='none', linewidth=2)
ax.set_xlim([0,2000])
ax.set_ylim([0,2000])
ax.invert_yaxis()

In [ ]:
slides

In [ ]:
from glob import glob
from sklearn.manifold import TSNE
import altair as alt

In [ ]:
slides = [i.removeprefix("scbench_processed/").removesuffix(".pt") for i in glob("scbench_processed/*.pt")]

In [ ]:
all_meta = []
all_images = []
all_viz_images = []


for s in slides:
    meta = pd.read_csv(f"scbench_processed/{s}.csv")
    all_meta.append(meta["annot_labels"])
    images = torch.load(f"scbench_processed/{s}.pt")
    all_images.append(torch.stack([transform(i) for i in images]))
    images_viz = np.load(f"scbench_processed/{s}_viz.npy")
    all_viz_images.append(images_viz)

In [ ]:
def conditional_sample(group):
    if len(group) > 8:
        return group.sample(n=8, replace=False).index
    return group.index


In [ ]:
meta.groupby(["annot_labels"]).apply(conditional_sample).explode().sort_values().tolist()

In [ ]:
all_meta = pd.concat(all_meta)
all_images = torch.cat(all_images)
all_images_viz = np.concatenate(all_viz_images)

In [ ]:
sorted(set(all_meta.tolist()))

In [ ]:
macrophages = all_images_viz[all_meta=="immune/macrophages"]

In [ ]:
for i in macrophages:
    plt.figure()
    plt.imshow(i)

In [ ]:
lymphocytes = all_images_viz[all_meta=="immune/lymphocytes"]

In [ ]:
for i in lymphocytes:
    plt.figure()
    plt.imshow(i)

In [ ]:
data = get_process_raw_strips_data("NIO_UM_1992.3").numpy()
im = transform(torch.tensor(np.stack((
                field_flatten(data[0, ...]),
                field_flatten(data[1, ...])
            )).clip(min=0, max=65535)))
Image.fromarray((einops.rearrange(adjust_brightness(adjust_contrast(im, 2), 3), "c h w -> h w c").numpy()*255).astype(np.uint8)).save("temp.png")


In [ ]:
db_root ="/nfs/turbo/umms-tocho/root_srh_db/UM/"
strip_names = get_strip_names(f"{db_root}/{"NIO_UM_549.7".replace('.', '/')}")

In [ ]:
strip_names["ch2"].iloc[0]

In [ ]:
# pixel space tSNE

In [ ]:
feat = einops.rearrange(all_images, "b c h w -> b (c h w)")

In [ ]:
xy = TSNE(n_components=2, perplexity=50).fit_transform(feat)

In [ ]:
min_vals = xy.min(axis=0)
xy_normalized = (xy - min_vals) / (xy.max(axis=0) - min_vals)
xy_normalized = xy_normalized * 1.8 -.9

In [ ]:
tsne_data = pd.DataFrame({"label": all_meta, "x": xy_normalized[:,0], "y": xy_normalized[:,1]})

In [ ]:
class_selection = alt.selection_point(fields=["label"], bind='legend')
op = alt.condition( class_selection, alt.value(1.0), alt.value(0.1))

bind_range = alt.binding_range(min=4, max=128, name='Size ')
param_size = alt.param(name='point_size', bind=bind_range, value=32)
        
tsne_unit_axis = alt.Axis(tickSize=0,
                                  values=[-1, -0.6, -0.2, 0.2, 0.6, 1],
                                  domain=False,
                                  labels=False,
                                  title="")

alt.Chart(tsne_data.sample(4000)).mark_point(filled=True,
                size=param_size).encode(
    x=alt.X("x", scale=alt.Scale(domain=[-1, 1]), axis=tsne_unit_axis),
    y=alt.Y("y", scale=alt.Scale(domain=[-1, 1]),  axis=tsne_unit_axis),
    color="label",
    opacity=op
).properties(
    width=700,
    height=700,
).add_params(
    class_selection,
    param_size
).configure_axis(
    labelFontSize=24,
    titleFontSize=24
).interactive()

In [ ]:
#image = process_read_srh(f"/nfs/turbo/umms-tocho/root_srh_db/UM/{slide_id.replace('.', '/')}/patches/{slide_id.replace('.', '-')}-2000_0_0_0.tif")
#new_crop = new_strip_im_raw[:, 2000:2300, :300].numpy()
#new_crop = torch.tensor(np.stack((field_flatten(new_crop[0, ...]),
#        field_flatten(new_crop[1, ...]))).clip(min=0, max=65535))
#print((new_crop - image).norm(dim=0).abs().max())
#fig, ax = plt.subplots(1,3, figsize=(10,12))
#ax[0].imshow(einops.rearrange(transform(new_crop), "c h w -> h w c"))
#ax[1].imshow((new_crop - image).norm(dim=0).abs() / 2**16)#, vmin=0, vmax=0.0001)
#ax[2].imshow(einops.rearrange(transform(image), "c h w -> h w c"))

In [ ]:
for i, m in match_exploded.iterrows():
    imc = m["im_centroids"]
    
    if m["polygons"]:
        polyviz = [translate(m["polygons"], xoff=-imc[0]+hcsz, yoff=-imc[1]+hcsz)]
    else:
        polyviz=[]
        
        
    fig, ax = plt.subplots(1,4)
    #ax[0].set_title((cx, cy))
    plot_polygons(annot_with_bars[imc[1]-hcsz:imc[1]+hcsz, imc[0]-hcsz:imc[0]+hcsz],
                  polyviz,
                  ax[0],
                  edgecolor='lime', facecolor='none', linewidth=2)

    plot_polygons(orig_with_bars[imc[1]-hcsz:imc[1]+hcsz, imc[0]-hcsz:imc[0]+hcsz],
                  polyviz,
                  ax[1],
                  edgecolor='lime', facecolor='none', linewidth=2)



    ax[2].imshow(einops.rearrange(adjust_brightness(adjust_contrast(transform(m["images"][0]), 2), 3), "c h w -> h w c"))
    plot_polygons(einops.rearrange(adjust_brightness(adjust_contrast(transform(m["images"][0]), 2), 3), "c h w -> h w c"),
                  polyviz,
                   ax[3],
                  edgecolor='yellow', facecolor='none', linewidth=2)


In [ ]:
patch_anchors_xyxy = torch.tensor(np.hstack([patch_origin_coords_xy, patch_origin_coords_xy+300]))
test = torchvision.utils.draw_bounding_boxes(
    torch.tensor(einops.rearrange(annot_with_bars, "h w c -> c h w")),
    patch_anchors_xyxy, width=2, colors="green"
)
test = torchvision.utils.draw_bounding_boxes(
    test,
    global_bbox, width=2, colors="yellow"
)
test = einops.rearrange(test, "c h w -> h w c")
Image.fromarray(test.numpy()).save("test_coord_map.png")

In [ ]:

test = torchvision.utils.draw_bounding_boxes(
    (einops.rearrange(new_strip_im, "h w c -> c h w") * 255).to(torch.uint8),
    patch_anchors_xyxy, width=2, colors="green"
)
test = torchvision.utils.draw_bounding_boxes(
    test,
    global_bbox, width=2, colors="yellow"
)
test = einops.rearrange(test, "c h w -> h w c")
Image.fromarray(test.numpy().astype(np.uint8)).save("test_coord_map2.png")